# Use OpenRouter With Google Agent-DK

Setup Prerequisite:

1. [Signup at OpenRouter](https://openrouter.ai/)
2. [Create an API Key](https://openrouter.ai/settings/keys)
2. Select a Free Model (you can continue as we are using a free model here)

## Free and Paid Models

The OpenRouter supports the latest DeepSeek V3 0324 and 50+ other models for free. Most of them support the defacto standard: OpenAI Chat Completion API.


If you are using a free model variant (with an ID ending in :free), then you will be limited to 20 requests per minute and 200 requests per day.

**See all Models List: https://openrouter.ai/models**

Note: OpenRouter do not charge anything extra at inference time.

## Rate Limiting and Crediting

There are a few rate limits that apply to certain types of requests, regardless of account status:

- Free limit: If you are using a free model variant (with an ID ending in :free), then you will be limited to 20 requests per minute and 200 requests per day.

If your account has a negative credit balance, you may see 402 errors, including for free models. Adding credits to put your balance above zero allows you to use those models again.

[Reference](https://openrouter.ai/docs/api-reference/limits)

In [7]:
import nest_asyncio
nest_asyncio.apply()

## Provider Config

In [8]:
from get_models_list import get_free_openrouter_models
MODELS = get_free_openrouter_models('gemini')
MODELS

['google/gemini-2.5-pro-exp-03-25',
 'google/gemini-2.0-flash-exp:free',
 'google/gemini-flash-1.5-8b-exp']

In [9]:
#Reference: https://openrouter.ai/docs/quickstart

import os
from google.adk.models.lite_llm import LiteLlm

BASE_URL = "https://openrouter.ai/api/v1"
API_KEY=os.getenv("OPENROUTER_API_KEY")
MODEL = MODELS[1]
model = LiteLlm(
    model=MODEL,
    base_url=BASE_URL,
    api_key=API_KEY,
)
# Some other free models on 26th March:
# https://openrouter.ai/deepseek/deepseek-chat-v3-0324:free
# https://openrouter.ai/google/gemini-2.5-pro-exp-03-25:free

## 1. Using the OpenRouter API directly

In [10]:
import requests
import json

response = requests.post(
  url=f"{BASE_URL}/chat/completions",
  headers={
    "Authorization": f"Bearer {API_KEY}",
  },
  data=json.dumps({
    "model": MODEL,
    "messages": [
      {
        "role": "user",
        "content": "What is the meaning of life?"
      }
    ]
  })
)

print(response.json())

{'id': 'gen-1746998951-WX7EK65YpeA4h0AKr7MA', 'provider': 'Google AI Studio', 'model': 'google/gemini-2.0-flash-exp:free', 'object': 'chat.completion', 'created': 1746998956, 'choices': [{'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'STOP', 'index': 0, 'message': {'role': 'assistant', 'content': "Ah, the big question! Philosophers, theologians, scientists, and individuals have pondered the meaning of life for centuries, and there's no single, universally accepted answer. It's a deeply personal and subjective inquiry. Here's a breakdown of different perspectives and approaches:\n\n**1. Philosophical Perspectives:**\n\n*   **Nihilism:** The belief that life is inherently without objective meaning, purpose, or intrinsic value.\n\n*   **Existentialism:** This emphasizes individual freedom and responsibility.  We are born into a meaningless world and must create our own meaning through our choices and actions. Key figures include Jean-Paul Sartre and Albert Camus.\n\n*

In [11]:
data = response.json()
data['choices'][0]['message']['content']

"Ah, the big question! Philosophers, theologians, scientists, and individuals have pondered the meaning of life for centuries, and there's no single, universally accepted answer. It's a deeply personal and subjective inquiry. Here's a breakdown of different perspectives and approaches:\n\n**1. Philosophical Perspectives:**\n\n*   **Nihilism:** The belief that life is inherently without objective meaning, purpose, or intrinsic value.\n\n*   **Existentialism:** This emphasizes individual freedom and responsibility.  We are born into a meaningless world and must create our own meaning through our choices and actions. Key figures include Jean-Paul Sartre and Albert Camus.\n\n*   **Absurdism:** Recognizing the inherent conflict between humanity's search for meaning and the universe's lack of it.  The focus shifts to how we respond to this absurdity.  (Often linked to Camus).\n\n*   **Hedonism:** The pursuit of pleasure and avoidance of pain as the primary goal in life.\n\n*   **Stoicism:** 

## 2. Using Google Agents DK

In [12]:
from google.adk.agents import Agent

root_agent = Agent(
    name="greeting_agent",
    # https://ai.google.dev/gemini-api/docs/models
    model=model,
    description="Greeting agent",
    instruction="""
    You are a helpful assistant that greets the user. 
    Ask for the user's name and greet them by name.
    """,
)


In [19]:
from google.adk.runners import Runner
